# Debug Eval DAG


Interactive notebook that mirrors the Airflow evaluation DAGs (`dags/eval_dags.py` → `experiments/scripts/eval/runner.py`).
Use this to step through dataset loading, gateway calls, and metric computation without running Airflow.

## 1. Setup Imports and Paths

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent  # experiments/notebooks → repo root
for p in [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

import httpx
from datasets import load_from_disk

## 2. Load Configuration

Set the same env vars visible inside the Airflow containers.
Override `EVAL_GATEWAY_URL` / `EVAL_BERT_SCORE_MODEL` / `EVAL_SAMPLE_LIMIT` as needed.

In [2]:
import os

# -- Adjust these to match your environment --
os.environ.setdefault("EVAL_GATEWAY_URL", "http://gateway:9000")
os.environ.setdefault("EVAL_SAMPLE_LIMIT", "5")  # small subset for debugging
os.environ.setdefault("EVAL_BERT_SCORE_MODEL", "microsoft/deberta-base-mnli")

from shared.config import get_eval_settings, get_settings

eval_cfg = get_eval_settings()
settings = get_settings()

print(f"Gateway URL   : {eval_cfg.gateway_url}")
print(f"BERTScore model: {eval_cfg.bert_score_model}")
print(f"Sample limit  : {eval_cfg.sample_limit}")
print(f"Base model    : {settings.default_model}")

ModuleNotFoundError: No module named 'shared'

## 3. Choose Eval Parameters

Pick a `(task, dataset, metric)` triple — the same unit the Airflow DAGs run.

In [3]:
TASK = "chat"
DATASET = "hotpotqa"
METRIC = "bertscore_f1"  # one of: relevance, correctness, bertscore_f1, rouge_l

RAG_ALIAS = "none"
LORA_ALIAS = "none"

# Valid metrics per task (for reference)
TASK_METRICS = {
    "chat": ["relevance", "correctness", "bertscore_f1", "rouge_l"],
    "summarize": ["faithfulness", "coverage", "bertscore_f1", "rouge_l"],
    "code": ["pass_at_1", "executable_rate"],
    "retrieval": ["recall_at_10", "ndcg_at_10"],
}
assert METRIC in TASK_METRICS[TASK], f"Invalid metric {METRIC} for task {TASK}"

## 4. Load Dataset

Load samples from the local Arrow files in `assets/datasets/`.
Same logic as `runner._load_dataset_samples()`.

In [5]:
ds_dict = load_from_disk(PROJECT_ROOT / "assets" / "datasets" / "hotpotqa")

In [16]:
samples = []

count = 0
for item in ds_dict['validation']:
    samples.append({
        'question': item.get('question', ''),
        'answer': item.get('answer')
    })
    count += 1
    if count >= 20:
        break

print(len(samples))

20


## 5. Generate Predictions via Gateway

Call the gateway chat-completions endpoint for each sample,
exactly as `runner._evaluate_generation()` does.

In [31]:
predictions: list[str] = []
references: list[str] = []

for i, sample in enumerate(samples):
    question = sample["question"]
    reference = sample.get("answer", "")

    payload = {
        "messages": [{"role": "user", "content": question}],
    }
    headers = {}
    headers["X-API-Key"] = 'asodih2!'

    resp = httpx.post(
        f"http://gateway:9000/v1/chat/completions",
        json=payload,
        headers=headers,
        timeout=120,
    )
    resp.raise_for_status()
    answer = resp.json()["choices"][0]["message"]["content"]

    predictions.append(answer)
    references.append(reference)
    print(f"[{i + 1}/{len(samples)}] {question[:80]}...")

print(f"\nGenerated {len(predictions)} predictions")

2026-03-19 17:42:08,806 httpx INFO HTTP Request: POST http://gateway:9000/v1/chat/completions "HTTP/1.1 200 OK"


[1/20] Were Scott Derrickson and Ed Wood of the same nationality?...


2026-03-19 17:42:10,921 httpx INFO HTTP Request: POST http://gateway:9000/v1/chat/completions "HTTP/1.1 200 OK"


[2/20] What government position was held by the woman who portrayed Corliss Archer in t...


2026-03-19 17:42:12,037 httpx INFO HTTP Request: POST http://gateway:9000/v1/chat/completions "HTTP/1.1 200 OK"


...

[18/20] Are Giuseppe Verdi and Ambroise Thomas both Opera composers ?...


2026-03-19 17:42:29,821 httpx INFO HTTP Request: POST http://gateway:9000/v1/chat/completions "HTTP/1.1 200 OK"


[19/20] Roger O. Egeberg was Assistant Secretary for Health and Scientific Affairs durin...


2026-03-19 17:42:30,563 httpx INFO HTTP Request: POST http://gateway:9000/v1/chat/completions "HTTP/1.1 200 OK"


[20/20] Which writer was from England, Henry Roth or Robert Erskine Childers?...

Generated 20 predictions


## 6. Compute Metric

Run the selected metric on the collected predictions/references.
This is the step that OOM'd with `deberta-xlarge-mnli` in Airflow.

In [33]:
from experiments.scripts.eval.metrics.automatic import compute_automatic_metrics

metrics = compute_automatic_metrics(
    predictions,
    references,
    bert_score_model='microsoft/deberta-base-mnli',
    metric='bertscore_f1',
)
metrics

ModuleNotFoundError: No module named 'bert_score'

## 7. Run Full Eval via `runner.run_eval()`

Alternatively, call the runner entry point directly — same function the Airflow DAG invokes.

In [ ]:
from experiments.scripts.eval.runner import run_eval

rows = run_eval(
    task=TASK,
    dataset_name=DATASET,
    metric=METRIC,
    rag_aliases=[RAG_ALIAS],
    lora_aliases=[LORA_ALIAS],
)
rows

## 8. Inspect Results

Show per-sample predictions vs references and the final metric value.

In [ ]:
import pandas as pd

# Per-sample comparison (from the step-by-step path)
df_samples = pd.DataFrame(
    {
        "question": [s["question"] for s in samples],
        "reference": references,
        "prediction": predictions,
    }
)
df_samples

In [ ]:
# Eval-run rows (from the full runner path)
if rows:
    df_results = pd.DataFrame(rows)
    display_cols = [
        "task",
        "dataset_name",
        "metric_name",
        "metric_value",
        "rag_alias",
        "lora_alias",
        "status",
    ]
    df_results[[c for c in display_cols if c in df_results.columns]]